In [2]:
import os
import re
import time
import sqlite3
import numpy as np
import pandas as pd
import requests

from tqdm import tqdm
from dotenv import load_dotenv

In [12]:
# 데이터 로드
RAW_PATH = "../data/makangs_raw_20260127_113115.csv"
ST_PATH  = "../data/서울교통공사_역주소 및 전화번호_20250318.csv"
CODE_PATH = "../data/sigungu_code_info.csv"

df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()

df["name"] = df["name_raw"]
df["category"] = df["category_raw"]

# 서교공 파일은 EUC-KR 인코딩
st_raw = pd.read_csv(ST_PATH, encoding="euc-kr")

print(df_raw.shape, df_raw.columns.tolist())
print(st_raw.shape, st_raw.columns.tolist())

(334, 10) ['source_site', 'crawled_at', 'listing_url', 'detail_url', 'external_id', 'name_raw', 'category_raw', 'address_raw', 'map_url_raw', 'status_raw']
(289, 7) ['연번', '역번호', '호선', '역명', '역전화번호', '도로명주소', '지번주소']


In [13]:
def normalize_spaces(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def normalize_seoul_prefix(addr):
    addr = normalize_spaces(addr)
    addr = re.sub(r"^서울시\s*", "서울특별시 ", addr)
    addr = re.sub(r"^서울\s+", "서울특별시 ", addr)
    return addr

def split_address_station(raw):
    if pd.isna(raw):
        return (np.nan, np.nan)

    raw = normalize_spaces(raw)

    # 슬래시 우선 분리
    parts = re.split(r"\s*/\s*", raw)

    address_std = np.nan
    station_detail = np.nan

    for p in parts:
        # 서울 + 구
        if re.search(r"^(서울|서울시|서울특별시)\s+\S+구", p) and pd.isna(address_std):
            address_std = normalize_seoul_prefix(p)

        # 역 단서
        elif re.search(r"(역|출구|도보|인근|근처)", p) and pd.isna(station_detail):
            station_detail = p

    # 슬래시 없는 경우
    if pd.isna(address_std) and re.search(r"^(서울|서울시|서울특별시)\s+\S+구", raw):
        address_std = normalize_seoul_prefix(raw)

    if pd.isna(station_detail) and re.search(r"(역|출구|도보|인근|근처)", raw):
        station_detail = raw

    return (address_std, station_detail)

df[["address_std", "station_detail"]] = df["address_raw"].apply(
    lambda x: pd.Series(split_address_station(x))
)

df[["address_raw","address_std","station_detail"]].head(15)

,address_raw,address_std,station_detail
0,서울특별시 마포구 서교동 (상세주소 문의) / 홍대입구역 1번 출구 도보 1분,서울특별시 마포구 서교동 (상세주소 문의),홍대입구역 1번 출구 도보 1분
1,서울 강남구 논현동 182-25번지 화랑닭발 4층,서울특별시 강남구 논현동 182-25번지 화랑닭발 4층,NaN
2,서울특별시 금천구 벚꽃로40길 3 / 가산동 42-16 / 가산디지털단지역 2번 출...,서울특별시 금천구 벚꽃로40길 3,가산디지털단지역 2번 출구 도보 3분
3,서울 서대문구 연세로 4길 19 3층,서울특별시 서대문구 연세로 4길 19 3층,NaN
4,서울 은평구 연서로211 하경빌딩 4층,서울특별시 은평구 연서로211 하경빌딩 4층,NaN
5,서울특별시 강남구 역삼로3길 17-6,서울특별시 강남구 역삼로3길 17-6,서울특별시 강남구 역삼로3길 17-6
6,"서울 종로구 종로10길14, 4층","서울특별시 종로구 종로10길14, 4층",NaN
7,서울특별시 강남구 강남대로84길 23,서울특별시 강남구 강남대로84길 23,NaN
8,서울특별시 은평구 진흥로 17 역촌동 43-47 2층,서울특별시 은평구 진흥로 17 역촌동 43-47 2층,서울특별시 은평구 진흥로 17 역촌동 43-47 2층
9,서울특별시 강남구 삼성로71길 32,서울특별시 강남구 삼성로71길 32,NaN


In [14]:
def extract_station(detail):
    if pd.isna(detail):
        return np.nan
    m = re.search(r"([가-힣0-9]+역)", detail)
    return m.group(1) if m else np.nan

def extract_station_exit(detail):
    if pd.isna(detail):
        return np.nan
    m_station = re.search(r"([가-힣0-9]+역)", detail)
    m_exit = re.search(r"(\d+)\s*번\s*출구", detail)
    if m_station and m_exit:
        return f"{m_station.group(1)} {m_exit.group(1)}번 출구"
    return np.nan

df["station"] = df["station_detail"].apply(extract_station)
df["station_exit"] = df["station_detail"].apply(extract_station_exit)

df[["station_detail","station","station_exit"]].head(10)

,station_detail,station,station_exit
0,홍대입구역 1번 출구 도보 1분,홍대입구역,홍대입구역 1번 출구
1,NaN,NaN,NaN
2,가산디지털단지역 2번 출구 도보 3분,가산디지털단지역,가산디지털단지역 2번 출구
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,서울특별시 강남구 역삼로3길 17-6,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,서울특별시 은평구 진흥로 17 역촌동 43-47 2층,NaN,NaN
9,NaN,NaN,NaN


In [15]:
def clean_station_name(name):
    n = normalize_spaces(name)
    n = re.sub(r"\(.*?\)", "", n).strip()
    if not n.endswith("역"):
        n += "역"
    return n

st = st_raw.copy()
st["station_std"] = st["역명"].apply(clean_station_name)
st["station_road_addr"] = st["도로명주소"].astype(str).apply(normalize_seoul_prefix)

st_master = (
    st.groupby("station_std", as_index=False)
      .agg(station_road_addr=("station_road_addr", "first"))
)

station_road_dict = dict(zip(st_master["station_std"], st_master["station_road_addr"]))

df["station_road_addr"] = df["station"].map(station_road_dict)

In [16]:
def is_valid_location_logic(raw):
    if pd.isna(raw):
        return False

    raw = normalize_spaces(raw)

    # 서비스 지역 나열 제거
    if re.search(r"(서울|경기|인천).*전지역", raw):
        return False

    has_gu = bool(re.search(r"(서울|서울시|서울특별시)\s+\S+구", raw))

    # 도로명/지번 단서
    has_detail = bool(
        re.search(r"(로\s*\d+|길\s*\d+|\d+-\d+|\d+번지)", raw)
    )

    station_only = bool(re.search(r"(역|출구|도보)", raw)) and not has_gu

    if has_gu and has_detail:
        return True

    if station_only:
        return False

    return False

df["is_valid_location"] = df["address_raw"].apply(is_valid_location_logic)

In [17]:
def parse_gu(addr):
    if pd.isna(addr):
        return np.nan
    m = re.search(r"서울특별시\s+([가-힣]+구)", addr)
    return m.group(1) if m else np.nan

df["gu"] = df["address_std"].apply(parse_gu)

code_df = pd.read_csv(CODE_PATH, encoding="utf-8", dtype=str)
seoul_gu = code_df[code_df["SIGUNGU_CD"].str.startswith("11")].copy()
gu_code_dict = dict(zip(seoul_gu["SIGUNGU_NM"], seoul_gu["SIGUNGU_CD"]))

df["COL_ADM_SE"] = df["gu"].map(gu_code_dict)

print("COL_ADM_SE 결측:", df["COL_ADM_SE"].isna().sum())

COL_ADM_SE 결측: 8


In [18]:
# 자치구 파싱
def parse_gu_from_addr(addr: str):
    if addr is None or (isinstance(addr, float) and np.isnan(addr)):
        return np.nan
    a = normalize_seoul_prefix(addr)
    m = re.search(r"서울특별시\s+(\S+구)\b", a)
    if m:
        return m.group(1)
    # 혹시 '서울 강남구' 형태가 남아있다면
    m2 = re.search(r"서울\s+(\S+구)\b", a)
    if m2:
        return m2.group(1)
    return np.nan

# 1차 gu 채우기(상세주소가 있으면 주소에서)
df["gu"] = df["address_std"].apply(parse_gu_from_addr)

In [19]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

def extract_gu_from_google(result):
    for comp in result.get("address_components", []):
        if "administrative_area_level_2" in comp["types"]:
            if comp["long_name"].endswith("구"):
                return comp["long_name"]
    return None

def geocode_cached(query):
    if not query:
        return (None, None, None, None)

    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": query,
        "key": GOOGLE_API_KEY,
        "language": "ko",
        "region": "kr"
    }

    r = requests.get(url, params=params)
    data = r.json()

    if data["status"] == "OK":
        result = data["results"][0]
        loc = result["geometry"]["location"]

        lat = loc["lat"]
        lng = loc["lng"]
        gu_geo = extract_gu_from_google(result)

        time.sleep(0.1)
        return lat, lng, gu_geo, result

    return (None, None, None, None)

In [20]:
df["lat"] = np.nan
df["lng"] = np.nan

for i, row in tqdm(df.iterrows(), total=len(df)):

    if row["is_valid_location"]:
        query = row["address_std"]
    else:
        if pd.notna(row["station_exit"]):
            query = "서울특별시 " + row["station_exit"]
        elif pd.notna(row["station_road_addr"]):
            query = row["station_road_addr"]
        else:
            query = None

    lat, lng, gu_geo, _ = geocode_cached(query)

    df.at[i, "lat"] = lat
    df.at[i, "lng"] = lng

    # gu 보완
    if pd.isna(row["gu"]) and gu_geo:
        df.at[i, "gu"] = gu_geo

100%|██████████| 334/334 [03:55<00:00,  1.42it/s]


In [21]:
# sigu 생성

def build_sigu(gu):
    if pd.isna(gu):
        return np.nan
    gu = str(gu).strip()
    if not gu:
        return np.nan
    return f"서울특별시 {gu}"

df["sigu"] = df["gu"].apply(build_sigu)

# 점검
print("sigu 결측:", df["sigu"].isna().sum())
display(df[["gu","sigu"]].drop_duplicates().sort_values("gu").head(20))

sigu 결측: 8


,gu,sigu
1,강남구,서울특별시 강남구
46,강동구,서울특별시 강동구
149,강북구,서울특별시 강북구
18,강서구,서울특별시 강서구
66,관악구,서울특별시 관악구
48,광진구,서울특별시 광진구
69,구로구,서울특별시 구로구
2,금천구,서울특별시 금천구
71,노원구,서울특별시 노원구
75,도봉구,서울특별시 도봉구


In [24]:
clean_cols = [
    "name",
    "category",
    "address_std",
    "gu",
    "sigu",
    "lat",
    "lng",
    "is_valid_location",
    "COL_ADM_SE",
    "station",
    "station_exit",
    "station_detail",
]

df_clean = df[clean_cols].copy()

OUT = "../data/clean_makangs.csv"
df_clean.to_csv(OUT, index=False, encoding="utf-8-sig")

print("Saved:", OUT)
display(df_clean.head(20))

Saved: ../data/clean_makangs.csv


,name,category,address_std,gu,sigu,lat,lng,is_valid_location,COL_ADM_SE,station,station_exit,station_detail
0,홍대입구,마사지,서울특별시 마포구 서교동 (상세주소 문의),마포구,서울특별시 마포구,37.557527,126.924467,False,11140,홍대입구역,홍대입구역 1번 출구,홍대입구역 1번 출구 도보 1분
1,논현왓포타이마사지,마사지,서울특별시 강남구 논현동 182-25번지 화랑닭발 4층,강남구,서울특별시 강남구,37.506906,127.024952,True,11230,NaN,NaN,NaN
2,두바이테라피,마사지,서울특별시 금천구 벚꽃로40길 3,금천구,서울특별시 금천구,37.482163,126.883148,True,11180,가산디지털단지역,가산디지털단지역 2번 출구,가산디지털단지역 2번 출구 도보 3분
3,타이365,마사지,서울특별시 서대문구 연세로 4길 19 3층,서대문구,서울특별시 서대문구,37.557395,126.937811,True,11130,NaN,NaN,NaN
4,풋앤조이,마사지,서울특별시 은평구 연서로211 하경빌딩 4층,은평구,서울특별시 은평구,37.617389,126.919140,True,11120,NaN,NaN,NaN
5,사월왁싱,왁싱,서울특별시 강남구 역삼로3길 17-6,강남구,서울특별시 강남구,37.494702,127.030596,True,11230,NaN,NaN,서울특별시 강남구 역삼로3길 17-6
6,시온타이,마사지,"서울특별시 종로구 종로10길14, 4층",종로구,서울특별시 종로구,37.569395,126.984301,True,11010,NaN,NaN,NaN
7,왁싱맨,왁싱,서울특별시 강남구 강남대로84길 23,강남구,서울특별시 강남구,37.497135,127.030377,True,11230,NaN,NaN,NaN
8,이모꾸비,왁싱,서울특별시 은평구 진흥로 17 역촌동 43-47 2층,은평구,서울특별시 은평구,37.605660,126.921636,True,11120,NaN,NaN,서울특별시 은평구 진흥로 17 역촌동 43-47 2층
9,업클로즈왁싱,왁싱,서울특별시 강남구 삼성로71길 32,강남구,서울특별시 강남구,37.501717,127.056292,True,11230,NaN,NaN,NaN


In [25]:
# 품질 보고
print("총 행:", len(df))
print("실제 업소 주소(TRUE):", df["is_valid_location"].sum())
print("역 기반 주소(FALSE):", (~df["is_valid_location"]).sum())
print("최종 위경도 확보:", df["lat"].notna().sum())
print("위경도 결측:", df["lat"].isna().sum())
print("gu 결측:", df["gu"].isna().sum())

print("\n🔎 역 기반인데 address_std가 있는 케이스 샘플")
display(df[(df["is_valid_location"]==False) & (df["address_std"].notna())][
    ["name","address_raw","address_std","station_detail","station"]
].head(20))

총 행: 334
실제 업소 주소(TRUE): 289
역 기반 주소(FALSE): 45
최종 위경도 확보: 321
위경도 결측: 13
gu 결측: 8

🔎 역 기반인데 address_std가 있는 케이스 샘플


,name,address_raw,address_std,station_detail,station
0,홍대입구,서울특별시 마포구 서교동 (상세주소 문의) / 홍대입구역 1번 출구 도보 1분,서울특별시 마포구 서교동 (상세주소 문의),홍대입구역 1번 출구 도보 1분,홍대입구역
125,스파마사지,서울특별시 송파구 오금로 / 방이동,서울특별시 송파구 오금로,NaN,NaN
153,티파니스웨디시,서울특별시 구로구 구로동,서울특별시 구로구 구로동,NaN,NaN
187,더봄),서울특별시 영등포구 국회대로 / 여의도동 (국회의사당역 4번 출구 인근),서울특별시 영등포구 국회대로,여의도동 (국회의사당역 4번 출구 인근),국회의사당역
189,야놀자스웨디시,서울특별시 관악구 신림동 (상세주소 문의) / 신림역 3번 출구 도보 3분,서울특별시 관악구 신림동 (상세주소 문의),신림역 3번 출구 도보 3분,신림역
190,도희샵,서울특별시 송파구 방이동 (올림픽로) / 몽촌토성역 3번 출구 도보 3분,서울특별시 송파구 방이동 (올림픽로),몽촌토성역 3번 출구 도보 3분,몽촌토성역
194,로단테테라피,서울특별시 강서구 마곡동 (상세 주소 문의),서울특별시 강서구 마곡동 (상세 주소 문의),NaN,NaN
196,러시아스웨디시,서울특별시 관악구 신림동 (상세주소 문의) / 신림역 8번 출구 도보 1분,서울특별시 관악구 신림동 (상세주소 문의),신림역 8번 출구 도보 1분,신림역
197,러시안스웨디시,서울특별시 서초구 서초동 (상세주소 문의) / 양재역 5분 거리,서울특별시 서초구 서초동 (상세주소 문의),양재역 5분 거리,양재역
199,듀원스웨디시,서울특별시 강남구 역삼동 / 역삼역 3번 출구 2분 거리 / 강남역 1번 출구 4분 거리,서울특별시 강남구 역삼동,역삼역 3번 출구 2분 거리,역삼역
